In [1]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, mean_absolute_error
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv1D, Conv2D, BatchNormalization, MaxPooling1D, MaxPooling2D,
                                     GlobalAveragePooling1D, GlobalAveragePooling2D, Dense, Reshape, Layer,
                                     Lambda, Add, Multiply, Dropout)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# ================= 1. 物理参数与环境配置 =================
TYPE_NAMES = ["SingleTone", "Chirp", "Pulse", "HoppingJam", "NoiseFM", 
              "NoiseAM", "Comb", "Mixed", "Satellite"]
FS = 200000 
TRUTH_MAP = {
    0: [500, 0], 1: [60000, 0], 2: [45000, 0], 3: [85000, 0],
    4: [70000, 0], 5: [35000, 0], 6: [65000, 0], 7: [80000, 0], 8: [0, 0]
}

# ================= 2. MBF 特征提取组件 (完整版) =================

class ResNetBlock1D(Layer):
    def __init__(self, filters, kernel_size, strides=1, **kwargs):
        super().__init__(**kwargs)
        self.conv1 = Conv1D(filters, kernel_size, strides=strides, padding='same', use_bias=False)
        self.bn1 = BatchNormalization()
        self.conv2 = Conv1D(filters, kernel_size, padding='same', use_bias=False)
        self.bn2 = BatchNormalization()
        self.shortcut = Conv1D(filters, 1, strides=strides, padding='same') if strides != 1 else Lambda(lambda x: x)
    def call(self, inputs):
        x = tf.nn.relu(self.bn1(self.conv1(inputs)))
        x = self.bn2(self.conv2(x))
        return tf.nn.relu(Add()([x, self.shortcut(inputs)]))

class ResNetBlock2D(Layer):
    def __init__(self, filters, kernel_size, strides=1, **kwargs):
        super().__init__(**kwargs)
        self.conv1 = Conv2D(filters, kernel_size, strides=strides, padding='same', use_bias=False)
        self.bn1 = BatchNormalization()
        self.conv2 = Conv2D(filters, kernel_size, padding='same', use_bias=False)
        self.bn2 = BatchNormalization()
        self.shortcut = Conv2D(filters, 1, strides=strides, padding='same') if strides != 1 else Lambda(lambda x: x)
    def call(self, inputs):
        x = tf.nn.relu(self.bn1(self.conv1(inputs)))
        x = self.bn2(self.conv2(x))
        return tf.nn.relu(Add()([x, self.shortcut(inputs)]))

def extract_stat_features(x):
    mu = tf.reduce_mean(x, axis=1, keepdims=True)
    sigma = tf.math.reduce_std(x, axis=1, keepdims=True)
    rms = tf.sqrt(tf.reduce_mean(tf.square(x), axis=1, keepdims=True))
    kurt = tf.reduce_mean(tf.pow((x - mu) / (sigma + 1e-8), 4), axis=1, keepdims=True) - 3.0
    return tf.concat([mu, sigma, rms, kurt], axis=1)

class PLELayer(Layer):
    def __init__(self, num_tasks=3, num_shared=2, num_specific=1, expert_dim=128, **kwargs):
        super().__init__(**kwargs)
        self.num_tasks, self.num_shared, self.num_specific, self.expert_dim = num_tasks, num_shared, num_specific, expert_dim
    def build(self, input_shape):
        self.shared_experts = [Dense(self.expert_dim, activation='relu') for _ in range(self.num_shared)]
        self.specific_experts = [[Dense(self.expert_dim, activation='relu') for _ in range(self.num_specific)] for _ in range(self.num_tasks)]
        self.gates = [Dense(self.num_shared + self.num_specific, activation='softmax') for _ in range(self.num_tasks)]
    def call(self, inputs):
        shared_outs = [ex(inputs) for ex in self.shared_experts]
        task_outputs = []
        for t in range(self.num_tasks):
            spec_outs = [ex(inputs) for ex in self.specific_experts[t]]
            all_experts = tf.stack(shared_outs + spec_outs, axis=1)
            gate_weights = tf.expand_dims(self.gates[t](inputs), axis=-1)
            task_outputs.append(tf.reduce_sum(all_experts * gate_weights, axis=1))
        return task_outputs

# ================= 3. 消融模型组装 (保留 MBF & PLE，不执行数据增强) =================

def build_mbf_plenet_sprint(L=1024, num_classes=9):
    inputs = Input(shape=(L,), name='input_signal')
    x_t = Reshape((L, 1))(inputs)
    
    f_time = GlobalAveragePooling1D()(ResNetBlock1D(64, 7, strides=2)(x_t))
    
    stft = Lambda(lambda x: tf.abs(tf.signal.stft(x, frame_length=128, frame_step=64)))(inputs)
    f_freq = GlobalAveragePooling2D()(ResNetBlock2D(64, (3,3), strides=2)(Reshape((-1, 65, 1))(stft)))
    
    f_temporal = GlobalAveragePooling1D()(BatchNormalization()(Conv1D(64, 3, padding='same', groups=64)(Conv1D(64, 1, padding='same')(x_t))))
    f_stat = Dense(64, activation='relu')(Lambda(extract_stat_features)(inputs))

    branches = [f_time, f_freq, f_temporal, f_stat]
    weights = [Dense(1, activation='sigmoid')(b) for b in branches]
    f_fused = Dense(128, activation='relu')(Add()([Multiply()([b, w]) for b, w in zip(branches, weights)]))

    ple_tasks = PLELayer(num_tasks=3, expert_dim=128)(f_fused)
    det_out = Dense(1, activation='sigmoid', name='det_out')(ple_tasks[0])
    cls_out = Dense(num_classes, activation='softmax', name='cls_out')(Dense(128, activation='relu')(ple_tasks[1]))
    reg_out = Dense(3, activation='linear', name='reg_out')(ple_tasks[2])

    model = Model(inputs, [det_out, cls_out, reg_out])
    model.compile(optimizer=Adam(2e-4),
                  loss={'det_out': 'binary_crossentropy', 'cls_out': 'sparse_categorical_crossentropy', 'reg_out': 'mse'},
                  loss_weights={'det_out': 0.8, 'cls_out': 2.0, 'reg_out': 0.5},
                  metrics={'det_out': 'accuracy', 'cls_out': 'accuracy'})
    return model

# ================= 4. 纯净数据载入与评估逻辑 =================

def load_v5_dataset(path):
    all_x, all_y, all_j = [], [], []
    files = sorted([f for f in os.listdir(path) if f.endswith('_X.npy')])
    for fx in files:
        match = re.search(r'jnr(-?\d+)', fx); jnr = float(match.group(1)) if match else 0.0
        x, y = np.load(os.path.join(path, fx)), np.load(os.path.join(path, fx.replace('_X.npy', '_Y.npy')))
        if x.ndim == 3: x = x[:, :, 0]
        all_x.append(x); all_y.append(y); all_j.append(np.full(len(y), jnr))
    X, Y, J = np.vstack(all_x), np.concatenate(all_y), np.concatenate(all_j)
    params = np.stack([np.array([TRUTH_MAP[l][0]/FS for l in Y]), 
                       np.array([TRUTH_MAP[l][1]/(FS/2) for l in Y]), 
                       np.clip((10**(J/10)-0.1)/(1000-0.1), 0, 1)], axis=1).astype(np.float32)
    # 严格 7:2:1 划分
    X_train, X_tmp, d_train, d_tmp, y_train, y_tmp, p_train, p_tmp, j_train, j_tmp = train_test_split(X, (Y<8).astype(float), Y, params, J, test_size=0.3, stratify=Y, random_state=42)
    X_val, X_test, d_val, d_test, y_val, y_test, p_val, p_test, j_val, j_test = train_test_split(X_tmp, d_tmp, y_tmp, p_tmp, j_tmp, test_size=1/3, stratify=y_tmp, random_state=42)
    return (X_train, d_train, y_train, p_train), (X_test, d_test, y_test, p_test)

def run_scientific_evaluation(model, X_test, d_true, y_true, p_true):
    preds = model.predict(X_test, batch_size=128)
    det_p, cls_p, reg_p = (preds[0] > 0.5).astype(int).ravel(), np.argmax(preds[1], 1), preds[2]
    
    print("\n" + "="*20 + " 📊 消融实验结果 (w/o Data Augmentation) " + "="*20)
    print(f"✅ 干扰检测准确率: {accuracy_score(d_true, det_p)*100:.2f}%")
    print(f"🎯 总体识别准确率: {accuracy_score(y_true, cls_p)*100:.2f}%")
    
    p_names = ['带宽 (BW)', '中心频率 (Fc)', '干扰强度 (JNR)']
    for i in range(3):
        rmse = np.sqrt(np.mean((p_true[:, i] - reg_p[:, i])**2))
        safe_denom = np.max(p_true[:, i]) - np.min(p_true[:, i])
        print(f"   🔹 {p_names[i]:<10}: MAE = {mean_absolute_error(p_true[:,i], reg_p[:,i]):.4f} | NRMSE = {rmse/safe_denom:.4f}")

def main():
    root_path = "/root/autodl-tmp/validate/0218/dataset_final_ready_v5"
    train_set, test_set = load_v5_dataset(root_path)
    
    model = build_mbf_plenet_sprint()
    # 【消融变更】：直接训练，不使用任何 Data Augmentation Generator
    print("🔥 启动消融实验：移除数据增强（直接使用原始样本训练）...")
    model.fit(train_set[0], {'det_out': train_set[1], 'cls_out': train_set[2], 'reg_out': train_set[3]},
              epochs=50, batch_size=128, verbose=1)

    run_scientific_evaluation(model, test_set[0], test_set[1], test_set[2], test_set[3])

if __name__ == "__main__": main()

2026-02-20 21:31:01.435093: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-20 21:31:01.500374: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-20 21:31:02.474155: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2026-02-20 21:31:06.579486: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1635] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 920 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090 D, pci bus id: 0000:39:00.0, compute cap

🔥 启动消融实验：移除数据增强（直接使用原始样本训练）...
Epoch 1/50


2026-02-20 21:31:13.113100: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:424] Loaded cuDNN version 8600
2026-02-20 21:31:13.737984: I tensorflow/compiler/xla/service/service.cc:169] XLA service 0x7f215308b840 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-20 21:31:13.738028: I tensorflow/compiler/xla/service/service.cc:177]   StreamExecutor device (0): NVIDIA GeForce RTX 4090 D, Compute Capability 8.9
2026-02-20 21:31:13.976632: W tensorflow/tsl/framework/bfc_allocator.cc:296] Allocator (GPU_0_bfc) ran out of memory trying to allocate 272.01MiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2026-02-20 21:31:14.208240: I ./tensorflow/compiler/jit/device_compiler.h:180] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-02-20 21:31:14.212994: I tensorflow/compiler/xla

745/745 [==============================] - 51s 44ms/step - loss: 2.5422 - det_out_loss: 0.0195 - cls_out_loss: 1.2589 - reg_out_loss: 0.0176 - det_out_accuracy: 1.0000 - cls_out_accuracy: 0.5577
Epoch 2/50
745/745 [==============================] - 30s 40ms/step - loss: 1.8787 - det_out_loss: 9.5030e-07 - cls_out_loss: 0.9368 - reg_out_loss: 0.0102 - det_out_accuracy: 1.0000 - cls_out_accuracy: 0.6744
Epoch 3/50
745/745 [==============================] - 29s 39ms/step - loss: 1.6390 - det_out_loss: 2.9322e-07 - cls_out_loss: 0.8174 - reg_out_loss: 0.0084 - det_out_accuracy: 1.0000 - cls_out_accuracy: 0.7122
Epoch 4/50
745/745 [==============================] - 29s 38ms/step - loss: 1.4726 - det_out_loss: 1.8718e-07 - cls_out_loss: 0.7345 - reg_out_loss: 0.0073 - det_out_accuracy: 1.0000 - cls_out_accuracy: 0.7382
Epoch 5/50
745/745 [==============================] - 29s 39ms/step - loss: 1.3550 - det_out_loss: 1.5561e-07 - cls_out_loss: 0.6758 - reg_out_loss: 0.0067 - det_out_accuracy:

/tmp/ipykernel_24680/3367421310.py:139: RuntimeWarning: divide by zero encountered in float_scalars
  print(f"   🔹 {p_names[i]:<10}: MAE = {mean_absolute_error(p_true[:,i], reg_p[:,i]):.4f} | NRMSE = {rmse/safe_denom:.4f}")
